# ML-05 — Feature Vector and Leakage/Privacy Check

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/RodionOm/Search-ranking-ml/blob/main/work/notebooks/w03_feature_leakage_check.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Build the feature vector

*Code that actually builds it — engineered features, categorical handling, fills.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import pandas as pd, numpy as np

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
y = (df["trend_direction"] == "down").astype(int)
groups = df["client_id"]                      # только для разбиения, НЕ признак

X = pd.DataFrame(index=df.index)

# трафик ДО окна метки: older = 90d - last - prev ; prev = как есть
for m in ["impressions", "clicks", "sessions"]:
    older = df[f"{m}_90d"] - df[f"{m}_last_30d"] - df[f"{m}_prev_30d"]
    X[f"log_{m}_older_30d"] = np.log1p(older)
    X[f"log_{m}_prev_30d"]  = np.log1p(df[f"{m}_prev_30d"])

# импульс до окна метки: prev против older
X["early_momentum"] = X["log_impressions_prev_30d"] - X["log_impressions_older_30d"]

# CTR только в prev-окне (в %, как колонка ctr)
X["ctr_prev_30d"] = np.where(df["impressions_prev_30d"] > 0,
                             df["clicks_prev_30d"] / df["impressions_prev_30d"] * 100, 0)

X["content_age_days"] = df["content_age_days"]

# размер текста: пропуски систематические -> флаг + 0
X["has_word_count"] = df["word_count"].notna().astype(int)
X["word_count"]     = df["word_count"].fillna(0)

# keyword-данные: пропуски по content_type -> флаг + 0
X["has_keyword_data"]  = df["search_volume"].notna().astype(int)
X["log_search_volume"] = np.log1p(df["search_volume"].fillna(0))
X["competition"]       = df["competition"].fillna(0)
X["cpc"]               = df["cpc"].fillna(0)

# категории -> one-hot
cats = df[["content_type", "main_intent"]].fillna("unknown")
X = pd.concat([X, pd.get_dummies(cats, dtype=int)], axis=1)

print(X.shape)                                # (30000, 23)
print("NaN left:", int(X.isna().sum().sum())) # 0

## 2. Feature notes (meaning, missing, categorical, available-when?)

*For each feature: what it means, how missing values are handled, and whether it exists BEFORE the moment you predict.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. The leakage hunt

*Attack your own features: label-derived columns, future windows, product flags. Show the test.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 4. What I excluded and why

*The list of fields you refused to use — with one line of why each.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.